## 6. 最小 RAG 实现

最小 RAG 包含三步：**检索 → 拼提示词 → 生成答案**。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None

import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")
model = SentenceTransformer(EMBEDDING_MODEL)

HEADER_KEYS = {1: "title", 2: "section", 3: "subsection"}


def split_by_markdown_headers(text):
    sections = []
    metadata = {}
    lines = []
    in_fence = False

    def save_section():
        content = "\n".join(lines).strip()
        body_lines = lines[1:] if lines and re.match(r"^ {0,3}#{1,3}\s+", lines[0]) else lines
        has_body = any(line.strip() and line.strip() != "---" for line in body_lines)
        if content and has_body:
            sections.append({"text": content, "metadata": metadata.copy()})

    for line in text.splitlines():
        if re.match(r"^ {0,3}(```|~~~)", line):
            in_fence = not in_fence
            lines.append(line)
            continue

        match = None if in_fence else re.match(r"^ {0,3}(#{1,3})\s+(.+)$", line)
        if not match:
            lines.append(line)
            continue

        save_section()
        lines = [line]
        level = len(match.group(1))
        metadata[HEADER_KEYS[level]] = match.group(2).strip()
        for deeper_level in range(level + 1, 4):
            metadata.pop(HEADER_KEYS[deeper_level], None)

    save_section()
    return sections

def split_long_text(text, max_chars=800, separators=("\n\n", "\n", "。", "；", "，", " ")):
    if len(text) <= max_chars:
        return [text]
    if not separators:
        return [text[index:index + max_chars] for index in range(0, len(text), max_chars)]

    separator, *remaining = separators
    if separator not in text:
        return split_long_text(text, max_chars, tuple(remaining))

    raw_parts = text.split(separator)
    parts = [
        part + separator if index < len(raw_parts) - 1 else part
        for index, part in enumerate(raw_parts)
    ]
    chunks = []
    current = ""
    for part in parts:
        candidate = current + part
        if len(candidate) <= max_chars:
            current = candidate
            continue
        if current:
            chunks.append(current)
        if len(part) <= max_chars:
            current = part
        else:
            chunks.extend(split_long_text(part, max_chars, tuple(remaining)))
            current = ""
    if current:
        chunks.append(current)
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def split_markdown(text, max_chars=800):
    chunks = []
    for section in split_by_markdown_headers(text):
        for part in split_long_text(section["text"], max_chars):
            chunks.append({"text": part, "metadata": section["metadata"].copy()})
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for chunk in split_markdown(path.read_text(encoding="utf-8")):
            missing_headings = [
                heading for heading in chunk["metadata"].values()
                if heading not in chunk["text"]
            ]
            searchable_text = "\n".join([*missing_headings, chunk["text"]])
            items.append({
                "source": path.relative_to(data_dir).as_posix(),
                "text": searchable_text,
                **chunk["metadata"],
            })
    return items


def build_index(chunks):
    vectors = model.encode([item["text"] for item in chunks], normalize_embeddings=True)
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="fashion_knowledge",
        vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE),
    )
    client.upload_points(
        collection_name="fashion_knowledge",
        points=[models.PointStruct(id=i, vector=vector.tolist(), payload=chunk) for i, (vector, chunk) in enumerate(zip(vectors, chunks))],
    )
    return client

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [2]:
chunks = load_chunks()
qdrant = build_index(chunks)

def search(question, top_k=4):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge", query=question_vector, limit=top_k
    ).points
    return [{**hit.payload, "score": hit.score} for hit in hits]

### 6.1. 检查检索结果与提示词

In [3]:
question = "SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？"
results = search(question, top_k=4)
context = "\n\n".join(f"[来源：{item['source']}]\n{item['text']}" for item in results)
prompt = f"""请只根据下面的资料回答问题。
资料没有答案时，请回答“现有资料无法回答”，不要猜测。

资料：
{context}

问题：{question}"""
print(prompt[:2500])

请只根据下面的资料回答问题。
资料没有答案时，请回答“现有资料无法回答”，不要猜测。

资料：
[来源：产品/瑜伽裤-YG301/产品规格.md]
瑜伽裤 SKU-YG301 技术规格书
2. 面料规格
### 2.2 面料结构

经编(Warp-knitted)双面布：面层平纹致密组织提供防透光基础；底层微毛圈组织经碳素磨毛处理提供Butter-soft触感。克重230 GSM，厚度0.45mm，透气率85 mm/s。

[来源：产品/瑜伽裤-YG301/产品规格.md]
瑜伽裤 SKU-YG301 技术规格书
2. 面料规格
### 2.1 纤维成分

75% Nylon 66 (锦纶/超细聚酰胺) + 25% Lycra Spandex (莱卡四面弹氨纶)。纱线40D/48F双面精密经编，克重230 GSM (±5g)。

Nylon 66熔点265°C(高于Nylon 6的220°C)，密度1.14 g/cm³，回潮率4.5%，吸湿快干。40D/48F: 40旦尼尔粗细，48根长丝束，单丝0.83旦尼尔超细柔软。

[来源：产品/瑜伽裤-YG301/产品规格.md]
瑜伽裤 SKU-YG301 技术规格书
4. 后整理工艺
### 4.2 染色

经轴染色Beam Dyeing 98°C常压，活性染料Reactive Dyes。色牢度: 耐洗4级、耐汗渍4-5级、耐光4级。主色Carbon Black (Pantone Black 6 C)。

[来源：产品/瑜伽裤-YG301/产品规格.md]
瑜伽裤 SKU-YG301 技术规格书
2. 面料规格
### 2.3 功能性能

| 指标 | 测试方法 | 标准 | 实测 |
| :--- | :--- | :--- | :--- |
| 防透光 | SGS Squat-Proof | 透光率≤5% | ≤2% (5级) |
| 弹性恢复 | 500次循环拉伸 | 恢复率≥90% | ≥95% |
| 抗起球 | ASTM D3512 | ≥3.5级 | 4.5级 |
| 透气率 | ISO 9237 | ≥50mm/s | 85mm/s |
| 吸湿速干 | AATCC 201 | 30min干燥率≥60% | 72% |

---

问题：SKU-YG301 瑜伽裤的面料成分和防透光要求是什么？


### 6.2. 生成答案

In [4]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "你是服饰箱包知识库助手，只根据资料回答，并指出依据的来源。"},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("未调用模型：请配置 API 后重新运行")

根据资料：

- 面料成分：75% Nylon 66（锦纶/超细聚酰胺）+ 25% Lycra Spandex（莱卡四面弹氨纶）。
- 防透光要求：按 SGS Squat-Proof 测试方法，透光率≤5%；实测透光率≤2%（5级）。

依据来源：产品/瑜伽裤-YG301/产品规格.md。
